In [1]:
import os
import requests
import getpass
import time
import base64
from tqdm.notebook import tqdm # Visual progress bar for Jupyter

# --- CONFIGURATION ---
# Smart path detection for Local Project Structure
if os.path.exists("../data"):
    DOWNLOAD_DIR = "../data/raw/nasa_chl_data"
else:
    DOWNLOAD_DIR = "data/raw/nasa_chl_data"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# --- PASTE YOUR LINKS HERE ---
# Paste your full list of NASA URLs between the triple quotes below.
# The script will automatically convert L3b (Binned) links to L3m (Mapped) if needed.
url_list = """
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20020801_20020831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20020901_20020930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021001_20021031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021101_20021130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021201_20021231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030101_20030131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030201_20030228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030301_20030331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030401_20030430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030501_20030531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030601_20030630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030701_20030731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030801_20030831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030901_20030930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031001_20031031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031101_20031130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031201_20031231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040101_20040131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040201_20040229.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040301_20040331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040401_20040430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040501_20040531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040601_20040630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040701_20040731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040801_20040831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040901_20040930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041001_20041031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041101_20041130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041201_20041231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050101_20050131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050201_20050228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050301_20050331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050401_20050430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050501_20050531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050601_20050630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050701_20050731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050801_20050831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050901_20050930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051001_20051031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051101_20051130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051201_20051231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060101_20060131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060201_20060228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060301_20060331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060401_20060430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060501_20060531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060601_20060630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060701_20060731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060801_20060831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060901_20060930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061001_20061031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061101_20061130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061201_20061231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070101_20070131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070201_20070228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070301_20070331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070401_20070430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070501_20070531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070601_20070630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070701_20070731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070801_20070831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070901_20070930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071001_20071031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071101_20071130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071201_20071231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080101_20080131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080201_20080229.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080301_20080331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080401_20080430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080501_20080531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080601_20080630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080701_20080731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080801_20080831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080901_20080930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081001_20081031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081101_20081130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081201_20081231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090101_20090131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090201_20090228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090301_20090331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090401_20090430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090501_20090531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090601_20090630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090701_20090731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090801_20090831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090901_20090930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091001_20091031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091101_20091130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091201_20091231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100101_20100131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100201_20100228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100301_20100331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100401_20100430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100501_20100531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100601_20100630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100701_20100731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100801_20100831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100901_20100930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101001_20101031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101101_20101130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101201_20101231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110101_20110131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110201_20110228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110301_20110331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110401_20110430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110501_20110531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110601_20110630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110701_20110731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110801_20110831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110901_20110930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111001_20111031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111101_20111130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111201_20111231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120101_20120131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120201_20120229.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120301_20120331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120401_20120430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120501_20120531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120601_20120630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120701_20120731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120801_20120831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120901_20120930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121001_20121031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121101_20121130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121201_20121231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130101_20130131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130201_20130228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130301_20130331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130401_20130430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130501_20130531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130601_20130630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130701_20130731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130801_20130831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130901_20130930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131001_20131031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131101_20131130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131201_20131231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140101_20140131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140201_20140228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140301_20140331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140401_20140430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140501_20140531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140601_20140630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140701_20140731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140801_20140831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140901_20140930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141001_20141031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141101_20141130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141201_20141231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150101_20150131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150201_20150228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150301_20150331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150401_20150430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150501_20150531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150601_20150630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150701_20150731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150801_20150831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150901_20150930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151001_20151031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151101_20151130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151201_20151231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160101_20160131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160201_20160229.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160301_20160331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160401_20160430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160501_20160531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160601_20160630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160701_20160731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160801_20160831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160901_20160930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161001_20161031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161101_20161130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161201_20161231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170101_20170131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170201_20170228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170301_20170331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170401_20170430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170501_20170531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170601_20170630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170701_20170731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170801_20170831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170901_20170930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171001_20171031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171101_20171130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171201_20171231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180101_20180131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180201_20180228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180301_20180331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180401_20180430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180501_20180531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180601_20180630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180701_20180731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180801_20180831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180901_20180930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181001_20181031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181101_20181130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181201_20181231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190101_20190131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190201_20190228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190301_20190331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190401_20190430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190501_20190531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190601_20190630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190701_20190731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190801_20190831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190901_20190930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191001_20191031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191101_20191130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191201_20191231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200101_20200131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200201_20200229.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200301_20200331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200401_20200430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200501_20200531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200601_20200630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200701_20200731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200801_20200831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200901_20200930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201001_20201031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201101_20201130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201201_20201231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210101_20210131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210201_20210228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210301_20210331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210401_20210430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210501_20210531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210601_20210630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210701_20210731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210801_20210831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210901_20210930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211001_20211031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211101_20211130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211201_20211231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220101_20220131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220201_20220228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220301_20220331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220401_20220430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220501_20220531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220601_20220630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220701_20220731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220801_20220831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220901_20220930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221001_20221031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221101_20221130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221201_20221231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230101_20230131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230201_20230228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230301_20230331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230401_20230430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230501_20230531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230601_20230630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230701_20230731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230801_20230831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230901_20230930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231001_20231031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231101_20231130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231201_20231231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240101_20240131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240201_20240229.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240301_20240331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240401_20240430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240501_20240531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240601_20240630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240701_20240731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240801_20240831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240901_20240930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241001_20241031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241101_20241130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241201_20241231.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250101_20250131.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250201_20250228.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250301_20250331.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250401_20250430.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250501_20250531.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250601_20250630.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250701_20250731.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250801_20250831.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250901_20250930.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251001_20251031.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251101_20251130.L3m.MO.CHL.chlor_a.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251201_20251231.L3m.MO.CHL.chlor_a.9km.NRT.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20260101_20260131.L3m.MO.CHL.chlor_a.9km.NRT.nc
"""

def download_file_robust(url, filename, session):
    filepath = os.path.join(DOWNLOAD_DIR, filename)
    
    # Resume Logic
    if os.path.exists(filepath):
        # Check if file is valid (>1MB)
        if os.path.getsize(filepath) > 1000000: 
            return "SKIPPED"
        else:
            os.remove(filepath)

    try:
        # 1. Initiate Request (Handle Redirects Manually for Auth)
        response = session.get(url, stream=True)
        
        # If redirected to login page (Earthdata), re-send request to the new URL
        if response.history and "earthdata" in response.url:
             response = session.get(response.url, stream=True)

        response.raise_for_status()
        
        # 2. Download with Progress Bar
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024 # 1MB chunks
        
        with open(filepath, 'wb') as f, tqdm(
            desc=filename,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
            leave=False,
            mininterval=0.5
        ) as bar:
            for chunk in response.iter_content(chunk_size=block_size):
                if chunk:
                    size = f.write(chunk)
                    bar.update(size)
        
        # 3. Validation
        if os.path.getsize(filepath) < 100000: # 100KB check
            return "FAIL_SIZE"
            
        return "SUCCESS"
        
    except Exception as e:
        return f"ERROR: {e}"

def main():
    print("--- NASA Chlorophyll Downloader (Local Manual List) ---")
    print(f"Target: {os.path.abspath(DOWNLOAD_DIR)}")
    
    # Credentials
    print("\nEnter your EarthData Credentials:")
    user = getpass.getpass("Username: ")
    password = getpass.getpass("Password: ")
    
    # SETUP SESSION WITH AUTH
    session = requests.Session()
    # Create Basic Auth Header manually to persist through redirects
    auth_str = f"{user}:{password}"
    b64_auth = base64.b64encode(auth_str.encode()).decode()
    session.headers.update({'Authorization': f"Basic {b64_auth}"})
    
    # Parse Links
    raw_urls = [line.strip() for line in url_list.strip().split('\n') if line.strip()]
    print(f"\nQueueing {len(raw_urls)} files...")
    
    success_count = 0
    
    # Main Loop
    for i, raw_url in enumerate(tqdm(raw_urls, desc="Total Progress")):
        
        # Auto-Fix URLs: Ensure we get L3m (Mapped) instead of L3b (Binned)
        # This fixes the issue if you accidentally pasted "L3b" links
        url = raw_url.replace(".L3b.", ".L3m.").replace(".MO.CHL.nc", ".MO.CHL.chlor_a.4km.nc")
        
        # Clean up double extension if it happens
        if url.endswith(".nc.nc"): url = url[:-3]
            
        filename = url.split('/')[-1]
        
        # Attempt Download
        status = download_file_robust(url, filename, session)
        
        if status == "SUCCESS":
            success_count += 1
        elif status == "FAIL_SIZE":
            print(f"\n[FAIL] {filename}: File too small (Auth failed)")
        elif "ERROR" in status:
            print(f"\n[FAIL] {filename}: Network Error")
            
        # Polite delay
        time.sleep(0.5)

    print(f"\n--- Done. {success_count}/{len(raw_urls)} files ready. ---")

if __name__ == "__main__":
    main()

--- NASA Chlorophyll Downloader (Local Manual List) ---
Target: C:\Users\tejsr\Downloads\BlueEco_Project\data\raw\nasa_chl_data

Enter your EarthData Credentials:
Username: ········
Password: ········

Queueing 282 files...


Total Progress:   0%|          | 0/282 [00:00<?, ?it/s]

AQUA_MODIS.20020801_20020831.L3m.MO.CHL.chlor_a.9km.nc:   0%|          | 0.00/14.8M [00:00<?, ?iB/s]

AQUA_MODIS.20020901_20020930.L3m.MO.CHL.chlor_a.9km.nc:   0%|          | 0.00/15.7M [00:00<?, ?iB/s]

AQUA_MODIS.20021001_20021031.L3m.MO.CHL.chlor_a.9km.nc:   0%|          | 0.00/15.2M [00:00<?, ?iB/s]

...

AQUA_MODIS.20251001_20251031.L3m.MO.CHL.chlor_a.9km.nc:   0%|          | 0.00/13.5M [00:00<?, ?iB/s]

AQUA_MODIS.20251101_20251130.L3m.MO.CHL.chlor_a.9km.nc:   0%|          | 0.00/12.7M [00:00<?, ?iB/s]

AQUA_MODIS.20251201_20251231.L3m.MO.CHL.chlor_a.9km.NRT.nc:   0%|          | 0.00/12.7M [00:00<?, ?iB/s]

AQUA_MODIS.20260101_20260131.L3m.MO.CHL.chlor_a.9km.NRT.nc:   0%|          | 0.00/13.3M [00:00<?, ?iB/s]


--- Done. 282/282 files ready. ---


In [2]:
import os
import requests
import getpass
import time
import base64
from tqdm.notebook import tqdm # Visual progress bar for Jupyter

# --- CONFIGURATION ---
# Smart path detection for Local Project Structure
if os.path.exists("../data"):
    DOWNLOAD_DIR = "../data/raw/nasa_sst_data"
else:
    DOWNLOAD_DIR = "data/raw/nasa_sst_data"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# --- SETTINGS ---
# Set this to True to automatically convert any 4km links you paste into 9km links
CONVERT_TO_9KM = True

# --- PASTE YOUR LINKS HERE ---
# Paste your full list of NASA SST URLs between the triple quotes below.
url_list = """
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20020801_20020831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20020901_20020930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021001_20021031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021101_20021130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021201_20021231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030101_20030131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030201_20030228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030301_20030331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030401_20030430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030501_20030531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030601_20030630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030701_20030731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030801_20030831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030901_20030930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031001_20031031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031101_20031130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031201_20031231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040101_20040131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040201_20040229.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040301_20040331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040401_20040430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040501_20040531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040601_20040630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040701_20040731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040801_20040831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040901_20040930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041001_20041031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041101_20041130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041201_20041231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050101_20050131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050201_20050228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050301_20050331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050401_20050430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050501_20050531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050601_20050630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050701_20050731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050801_20050831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050901_20050930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051001_20051031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051101_20051130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051201_20051231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060101_20060131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060201_20060228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060301_20060331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060401_20060430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060501_20060531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060601_20060630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060701_20060731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060801_20060831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060901_20060930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061001_20061031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061101_20061130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061201_20061231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070101_20070131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070201_20070228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070301_20070331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070401_20070430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070501_20070531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070601_20070630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070701_20070731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070801_20070831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070901_20070930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071001_20071031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071101_20071130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071201_20071231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080101_20080131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080201_20080229.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080301_20080331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080401_20080430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080501_20080531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080601_20080630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080701_20080731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080801_20080831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080901_20080930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081001_20081031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081101_20081130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081201_20081231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090101_20090131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090201_20090228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090301_20090331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090401_20090430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090501_20090531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090601_20090630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090701_20090731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090801_20090831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090901_20090930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091001_20091031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091101_20091130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091201_20091231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100101_20100131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100201_20100228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100301_20100331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100401_20100430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100501_20100531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100601_20100630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100701_20100731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100801_20100831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100901_20100930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101001_20101031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101101_20101130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101201_20101231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110101_20110131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110201_20110228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110301_20110331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110401_20110430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110501_20110531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110601_20110630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110701_20110731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110801_20110831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110901_20110930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111001_20111031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111101_20111130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111201_20111231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120101_20120131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120201_20120229.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120301_20120331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120401_20120430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120501_20120531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120601_20120630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120701_20120731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120801_20120831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120901_20120930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121001_20121031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121101_20121130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121201_20121231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130101_20130131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130201_20130228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130301_20130331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130401_20130430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130501_20130531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130601_20130630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130701_20130731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130801_20130831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130901_20130930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131001_20131031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131101_20131130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131201_20131231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140101_20140131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140201_20140228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140301_20140331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140401_20140430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140501_20140531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140601_20140630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140701_20140731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140801_20140831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140901_20140930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141001_20141031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141101_20141130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141201_20141231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150101_20150131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150201_20150228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150301_20150331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150401_20150430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150501_20150531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150601_20150630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150701_20150731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150801_20150831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150901_20150930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151001_20151031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151101_20151130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151201_20151231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160101_20160131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160201_20160229.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160301_20160331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160401_20160430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160501_20160531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160601_20160630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160701_20160731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160801_20160831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160901_20160930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161001_20161031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161101_20161130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161201_20161231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170101_20170131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170201_20170228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170301_20170331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170401_20170430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170501_20170531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170601_20170630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170701_20170731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170801_20170831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170901_20170930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171001_20171031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171101_20171130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171201_20171231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180101_20180131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180201_20180228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180301_20180331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180401_20180430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180501_20180531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180601_20180630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180701_20180731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180801_20180831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180901_20180930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181001_20181031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181101_20181130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181201_20181231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190101_20190131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190201_20190228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190301_20190331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190401_20190430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190501_20190531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190601_20190630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190701_20190731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190801_20190831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190901_20190930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191001_20191031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191101_20191130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191201_20191231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200101_20200131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200201_20200229.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200301_20200331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200401_20200430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200501_20200531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200601_20200630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200701_20200731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200801_20200831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200901_20200930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201001_20201031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201101_20201130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201201_20201231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210101_20210131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210201_20210228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210301_20210331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210401_20210430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210501_20210531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210601_20210630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210701_20210731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210801_20210831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210901_20210930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211001_20211031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211101_20211130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211201_20211231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220101_20220131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220201_20220228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220301_20220331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220401_20220430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220501_20220531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220601_20220630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220701_20220731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220801_20220831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220901_20220930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221001_20221031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221101_20221130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221201_20221231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230101_20230131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230201_20230228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230301_20230331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230401_20230430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230501_20230531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230601_20230630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230701_20230731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230801_20230831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230901_20230930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231001_20231031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231101_20231130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231201_20231231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240101_20240131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240201_20240229.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240301_20240331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240401_20240430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240501_20240531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240601_20240630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240701_20240731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240801_20240831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240901_20240930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241001_20241031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241101_20241130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241201_20241231.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250101_20250131.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250201_20250228.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250301_20250331.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250401_20250430.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250501_20250531.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250601_20250630.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250701_20250731.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250801_20250831.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250901_20250930.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251001_20251031.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251101_20251130.L3m.MO.SST.sst.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251201_20251231.L3m.MO.SST.sst.9km.NRT.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20260101_20260131.L3m.MO.SST.sst.9km.NRT.nc
"""

def download_file_robust(url, filename, session):
    filepath = os.path.join(DOWNLOAD_DIR, filename)
    
    # Resume Logic
    if os.path.exists(filepath):
        # Check if file is valid (>1MB)
        if os.path.getsize(filepath) > 1000000: 
            return "SKIPPED"
        else:
            os.remove(filepath)

    try:
        # 1. Initiate Request (Handle Redirects Manually for Auth)
        response = session.get(url, stream=True)
        
        # If redirected to login page (Earthdata), re-send request to the new URL
        if response.history and "earthdata" in response.url:
             response = session.get(response.url, stream=True)

        response.raise_for_status()
        
        # 2. Download with Progress Bar
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024 # 1MB chunks
        
        with open(filepath, 'wb') as f, tqdm(
            desc=filename,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
            leave=False,
            mininterval=0.5
        ) as bar:
            for chunk in response.iter_content(chunk_size=block_size):
                if chunk:
                    size = f.write(chunk)
                    bar.update(size)
        
        # 3. Validation
        if os.path.getsize(filepath) < 100000: # 100KB check
            return "FAIL_SIZE"
            
        return "SUCCESS"
        
    except Exception as e:
        return f"ERROR: {e}"

def main():
    print("--- NASA SST Downloader (Local Manual List) ---")
    print(f"Target: {os.path.abspath(DOWNLOAD_DIR)}")
    print(f"Resolution Conversion: {'4km -> 9km' if CONVERT_TO_9KM else 'Keep Original'}")
    
    # Credentials
    print("\nEnter your EarthData Credentials:")
    user = getpass.getpass("Username: ")
    password = getpass.getpass("Password: ")
    
    # SETUP SESSION WITH AUTH
    session = requests.Session()
    # Create Basic Auth Header manually to persist through redirects
    auth_str = f"{user}:{password}"
    b64_auth = base64.b64encode(auth_str.encode()).decode()
    session.headers.update({'Authorization': f"Basic {b64_auth}"})
    
    # Parse Links
    raw_urls = [line.strip() for line in url_list.strip().split('\n') if line.strip()]
    print(f"\nQueueing {len(raw_urls)} files...")
    
    success_count = 0
    
    # Main Loop
    for i, raw_url in enumerate(tqdm(raw_urls, desc="Total Progress")):
        
        url = raw_url
        
        # Feature: Convert 4km links to 9km on the fly
        if CONVERT_TO_9KM and "4km" in url:
            url = url.replace("4km", "9km")
        
        # Auto-Fix URLs: Ensure we get L3m (Mapped) instead of L3b (Binned) if pasted wrong
        if ".L3b." in url:
            res_str = "9km" if CONVERT_TO_9KM else "4km"
            url = url.replace(".L3b.", ".L3m.").replace(".MO.SST.nc", f".MO.SST.sst.{res_str}.nc")
            
        filename = url.split('/')[-1]
        
        # Attempt Download
        status = download_file_robust(url, filename, session)
        
        if status == "SUCCESS":
            success_count += 1
        elif status == "FAIL_SIZE":
            print(f"\n[FAIL] {filename}: File too small (Auth failed)")
        elif "ERROR" in status:
            print(f"\n[FAIL] {filename}: Network Error")
            
        # Polite delay
        time.sleep(0.5)

    print(f"\n--- Done. {success_count}/{len(raw_urls)} files ready. ---")

if __name__ == "__main__":
    main()

--- NASA SST Downloader (Local Manual List) ---
Target: C:\Users\tejsr\Downloads\BlueEco_Project\data\raw\nasa_sst_data
Resolution Conversion: 4km -> 9km

Enter your EarthData Credentials:
Username: ········
Password: ········

Queueing 282 files...


Total Progress:   0%|          | 0/282 [00:00<?, ?it/s]

AQUA_MODIS.20020801_20020831.L3m.MO.SST.sst.9km.nc:   0%|          | 0.00/8.12M [00:00<?, ?iB/s]

AQUA_MODIS.20020901_20020930.L3m.MO.SST.sst.9km.nc:   0%|          | 0.00/7.75M [00:00<?, ?iB/s]

AQUA_MODIS.20021001_20021031.L3m.MO.SST.sst.9km.nc:   0%|          | 0.00/7.54M [00:00<?, ?iB/s]

...

AQUA_MODIS.20251001_20251031.L3m.MO.SST.sst.9km.nc:   0%|          | 0.00/6.52M [00:00<?, ?iB/s]

AQUA_MODIS.20251101_20251130.L3m.MO.SST.sst.9km.nc:   0%|          | 0.00/6.29M [00:00<?, ?iB/s]

AQUA_MODIS.20251201_20251231.L3m.MO.SST.sst.9km.NRT.nc:   0%|          | 0.00/6.18M [00:00<?, ?iB/s]

AQUA_MODIS.20260101_20260131.L3m.MO.SST.sst.9km.NRT.nc:   0%|          | 0.00/6.19M [00:00<?, ?iB/s]


--- Done. 282/282 files ready. ---


In [1]:
import os
import requests
import getpass
import time
import base64
from tqdm.notebook import tqdm # Visual progress bar for Jupyter

# --- CONFIGURATION ---
# Smart path detection for Local Project Structure
if os.path.exists("../data"):
    DOWNLOAD_DIR = "../data/raw/nasa_par_data" # <--- CHANGED FOR PAR
else:
    DOWNLOAD_DIR = "data/raw/nasa_par_data"    # <--- CHANGED FOR PAR

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# --- SETTINGS ---
# Set this to True to automatically convert any 4km links you paste into 9km links
CONVERT_TO_9KM = True

# --- PASTE YOUR LINKS HERE ---
# Paste your full list of NASA PAR URLs between the triple quotes below.
url_list = """
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20020801_20020831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20020901_20020930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021001_20021031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021101_20021130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021201_20021231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030101_20030131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030201_20030228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030301_20030331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030401_20030430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030501_20030531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030601_20030630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030701_20030731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030801_20030831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030901_20030930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031001_20031031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031101_20031130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031201_20031231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040101_20040131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040201_20040229.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040301_20040331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040401_20040430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040501_20040531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040601_20040630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040701_20040731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040801_20040831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040901_20040930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041001_20041031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041101_20041130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041201_20041231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050101_20050131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050201_20050228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050301_20050331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050401_20050430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050501_20050531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050601_20050630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050701_20050731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050801_20050831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050901_20050930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051001_20051031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051101_20051130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051201_20051231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060101_20060131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060201_20060228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060301_20060331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060401_20060430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060501_20060531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060601_20060630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060701_20060731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060801_20060831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060901_20060930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061001_20061031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061101_20061130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061201_20061231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070101_20070131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070201_20070228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070301_20070331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070401_20070430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070501_20070531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070601_20070630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070701_20070731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070801_20070831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070901_20070930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071001_20071031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071101_20071130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071201_20071231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080101_20080131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080201_20080229.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080301_20080331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080401_20080430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080501_20080531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080601_20080630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080701_20080731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080801_20080831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080901_20080930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081001_20081031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081101_20081130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081201_20081231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090101_20090131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090201_20090228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090301_20090331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090401_20090430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090501_20090531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090601_20090630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090701_20090731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090801_20090831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090901_20090930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091001_20091031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091101_20091130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091201_20091231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100101_20100131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100201_20100228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100301_20100331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100401_20100430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100501_20100531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100601_20100630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100701_20100731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100801_20100831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100901_20100930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101001_20101031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101101_20101130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101201_20101231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110101_20110131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110201_20110228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110301_20110331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110401_20110430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110501_20110531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110601_20110630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110701_20110731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110801_20110831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110901_20110930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111001_20111031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111101_20111130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111201_20111231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120101_20120131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120201_20120229.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120301_20120331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120401_20120430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120501_20120531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120601_20120630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120701_20120731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120801_20120831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120901_20120930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121001_20121031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121101_20121130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121201_20121231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130101_20130131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130201_20130228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130301_20130331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130401_20130430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130501_20130531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130601_20130630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130701_20130731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130801_20130831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130901_20130930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131001_20131031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131101_20131130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131201_20131231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140101_20140131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140201_20140228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140301_20140331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140401_20140430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140501_20140531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140601_20140630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140701_20140731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140801_20140831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140901_20140930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141001_20141031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141101_20141130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141201_20141231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150101_20150131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150201_20150228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150301_20150331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150401_20150430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150501_20150531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150601_20150630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150701_20150731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150801_20150831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150901_20150930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151001_20151031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151101_20151130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151201_20151231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160101_20160131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160201_20160229.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160301_20160331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160401_20160430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160501_20160531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160601_20160630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160701_20160731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160801_20160831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160901_20160930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161001_20161031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161101_20161130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161201_20161231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170101_20170131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170201_20170228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170301_20170331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170401_20170430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170501_20170531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170601_20170630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170701_20170731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170801_20170831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170901_20170930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171001_20171031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171101_20171130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171201_20171231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180101_20180131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180201_20180228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180301_20180331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180401_20180430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180501_20180531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180601_20180630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180701_20180731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180801_20180831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180901_20180930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181001_20181031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181101_20181130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181201_20181231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190101_20190131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190201_20190228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190301_20190331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190401_20190430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190501_20190531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190601_20190630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190701_20190731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190801_20190831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190901_20190930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191001_20191031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191101_20191130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191201_20191231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200101_20200131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200201_20200229.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200301_20200331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200401_20200430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200501_20200531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200601_20200630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200701_20200731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200801_20200831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200901_20200930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201001_20201031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201101_20201130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201201_20201231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210101_20210131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210201_20210228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210301_20210331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210401_20210430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210501_20210531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210601_20210630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210701_20210731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210801_20210831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210901_20210930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211001_20211031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211101_20211130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211201_20211231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220101_20220131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220201_20220228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220301_20220331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220401_20220430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220501_20220531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220601_20220630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220701_20220731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220801_20220831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220901_20220930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221001_20221031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221101_20221130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221201_20221231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230101_20230131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230201_20230228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230301_20230331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230401_20230430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230501_20230531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230601_20230630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230701_20230731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230801_20230831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230901_20230930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231001_20231031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231101_20231130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231201_20231231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240101_20240131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240201_20240229.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240301_20240331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240401_20240430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240501_20240531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240601_20240630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240701_20240731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240801_20240831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240901_20240930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241001_20241031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241101_20241130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241201_20241231.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250101_20250131.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250201_20250228.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250301_20250331.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250401_20250430.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250501_20250531.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250601_20250630.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250701_20250731.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250801_20250831.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250901_20250930.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251001_20251031.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251101_20251130.L3m.MO.PAR.par.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251201_20251231.L3m.MO.PAR.par.9km.NRT.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20260101_20260131.L3m.MO.PAR.par.9km.NRT.nc
"""

def download_file_robust(url, filename, session):
    filepath = os.path.join(DOWNLOAD_DIR, filename)
    
    # Resume Logic
    if os.path.exists(filepath):
        # Check if file is valid (>1MB)
        if os.path.getsize(filepath) > 1000000: 
            return "SKIPPED"
        else:
            os.remove(filepath)

    try:
        # 1. Initiate Request (Handle Redirects Manually for Auth)
        response = session.get(url, stream=True)
        
        # If redirected to login page (Earthdata), re-send request to the new URL
        if response.history and "earthdata" in response.url:
             response = session.get(response.url, stream=True)

        response.raise_for_status()
        
        # 2. Download with Progress Bar
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024 # 1MB chunks
        
        with open(filepath, 'wb') as f, tqdm(
            desc=filename,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
            leave=False,
            mininterval=0.5
        ) as bar:
            for chunk in response.iter_content(chunk_size=block_size):
                if chunk:
                    size = f.write(chunk)
                    bar.update(size)
        
        # 3. Validation
        if os.path.getsize(filepath) < 100000: # 100KB check
            return "FAIL_SIZE"
            
        return "SUCCESS"
        
    except Exception as e:
        return f"ERROR: {e}"

def main():
    print("--- NASA PAR Downloader (Local Manual List) ---")
    print(f"Target: {os.path.abspath(DOWNLOAD_DIR)}")
    print(f"Resolution Conversion: {'4km -> 9km' if CONVERT_TO_9KM else 'Keep Original'}")
    
    # Credentials
    print("\nEnter your EarthData Credentials:")
    user = getpass.getpass("Username: ")
    password = getpass.getpass("Password: ")
    
    # SETUP SESSION WITH AUTH
    session = requests.Session()
    # Create Basic Auth Header manually to persist through redirects
    auth_str = f"{user}:{password}"
    b64_auth = base64.b64encode(auth_str.encode()).decode()
    session.headers.update({'Authorization': f"Basic {b64_auth}"})
    
    # Parse Links
    raw_urls = [line.strip() for line in url_list.strip().split('\n') if line.strip()]
    print(f"\nQueueing {len(raw_urls)} files...")
    
    success_count = 0
    
    # Main Loop
    for i, raw_url in enumerate(tqdm(raw_urls, desc="Total Progress")):
        
        url = raw_url
        
        # Feature: Convert 4km links to 9km on the fly
        if CONVERT_TO_9KM and "4km" in url:
            url = url.replace("4km", "9km")
        
        # NOTE: Removed SST-specific .L3b fixer here to prevent errors with PAR filenames
            
        filename = url.split('/')[-1]
        
        # Attempt Download
        status = download_file_robust(url, filename, session)
        
        if status == "SUCCESS":
            success_count += 1
        elif status == "FAIL_SIZE":
            print(f"\n[FAIL] {filename}: File too small (Auth failed)")
        elif "ERROR" in status:
            print(f"\n[FAIL] {filename}: Network Error")
            
        # Polite delay
        time.sleep(0.5)

    print(f"\n--- Done. {success_count}/{len(raw_urls)} files ready. ---")

if __name__ == "__main__":
    main()

--- NASA PAR Downloader (Local Manual List) ---
Target: C:\Users\tejsr\Downloads\BlueEco_Project\data\raw\nasa_par_data
Resolution Conversion: 4km -> 9km

Enter your EarthData Credentials:


Username:  ········
Password:  ········



Queueing 282 files...


Total Progress:   0%|          | 0/282 [00:00<?, ?it/s]

AQUA_MODIS.20020801_20020831.L3m.MO.PAR.par.9km.nc:   0%|          | 0.00/9.88M [00:00<?, ?iB/s]

...

AQUA_MODIS.20251001_20251031.L3m.MO.PAR.par.9km.nc:   0%|          | 0.00/8.46M [00:00<?, ?iB/s]

AQUA_MODIS.20251101_20251130.L3m.MO.PAR.par.9km.nc:   0%|          | 0.00/8.01M [00:00<?, ?iB/s]

AQUA_MODIS.20251201_20251231.L3m.MO.PAR.par.9km.NRT.nc:   0%|          | 0.00/7.78M [00:00<?, ?iB/s]

AQUA_MODIS.20260101_20260131.L3m.MO.PAR.par.9km.NRT.nc:   0%|          | 0.00/7.83M [00:00<?, ?iB/s]


--- Done. 282/282 files ready. ---


In [1]:
import os
import requests
import getpass
import time
import base64
from tqdm.notebook import tqdm # Visual progress bar for Jupyter

# --- CONFIGURATION ---
# Smart path detection for Local Project Structure
if os.path.exists("../data"):
    DOWNLOAD_DIR = "../data/raw/nasa_kd490_data" # <--- CHANGED FOR KD_490
else:
    DOWNLOAD_DIR = "data/raw/nasa_kd490_data"    # <--- CHANGED FOR KD_490

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# --- SETTINGS ---
# Set this to True to automatically convert any 4km links you paste into 9km links
CONVERT_TO_9KM = True

# --- PASTE YOUR LINKS HERE ---
# Paste your full list of NASA Kd_490 URLs between the triple quotes below.
url_list = """
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20020801_20020831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20020901_20020930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021001_20021031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021101_20021130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021201_20021231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030101_20030131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030201_20030228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030301_20030331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030401_20030430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030501_20030531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030601_20030630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030701_20030731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030801_20030831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030901_20030930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031001_20031031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031101_20031130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031201_20031231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040101_20040131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040201_20040229.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040301_20040331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040401_20040430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040501_20040531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040601_20040630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040701_20040731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040801_20040831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040901_20040930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041001_20041031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041101_20041130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041201_20041231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050101_20050131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050201_20050228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050301_20050331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050401_20050430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050501_20050531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050601_20050630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050701_20050731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050801_20050831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050901_20050930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051001_20051031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051101_20051130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051201_20051231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060101_20060131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060201_20060228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060301_20060331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060401_20060430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060501_20060531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060601_20060630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060701_20060731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060801_20060831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060901_20060930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061001_20061031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061101_20061130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061201_20061231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070101_20070131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070201_20070228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070301_20070331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070401_20070430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070501_20070531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070601_20070630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070701_20070731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070801_20070831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070901_20070930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071001_20071031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071101_20071130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071201_20071231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080101_20080131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080201_20080229.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080301_20080331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080401_20080430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080501_20080531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080601_20080630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080701_20080731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080801_20080831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080901_20080930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081001_20081031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081101_20081130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081201_20081231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090101_20090131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090201_20090228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090301_20090331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090401_20090430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090501_20090531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090601_20090630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090701_20090731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090801_20090831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090901_20090930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091001_20091031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091101_20091130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091201_20091231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100101_20100131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100201_20100228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100301_20100331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100401_20100430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100501_20100531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100601_20100630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100701_20100731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100801_20100831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100901_20100930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101001_20101031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101101_20101130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101201_20101231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110101_20110131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110201_20110228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110301_20110331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110401_20110430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110501_20110531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110601_20110630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110701_20110731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110801_20110831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110901_20110930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111001_20111031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111101_20111130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111201_20111231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120101_20120131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120201_20120229.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120301_20120331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120401_20120430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120501_20120531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120601_20120630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120701_20120731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120801_20120831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120901_20120930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121001_20121031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121101_20121130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121201_20121231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130101_20130131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130201_20130228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130301_20130331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130401_20130430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130501_20130531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130601_20130630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130701_20130731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130801_20130831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130901_20130930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131001_20131031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131101_20131130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131201_20131231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140101_20140131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140201_20140228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140301_20140331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140401_20140430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140501_20140531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140601_20140630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140701_20140731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140801_20140831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140901_20140930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141001_20141031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141101_20141130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141201_20141231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150101_20150131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150201_20150228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150301_20150331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150401_20150430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150501_20150531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150601_20150630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150701_20150731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150801_20150831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150901_20150930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151001_20151031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151101_20151130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151201_20151231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160101_20160131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160201_20160229.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160301_20160331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160401_20160430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160501_20160531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160601_20160630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160701_20160731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160801_20160831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160901_20160930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161001_20161031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161101_20161130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161201_20161231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170101_20170131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170201_20170228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170301_20170331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170401_20170430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170501_20170531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170601_20170630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170701_20170731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170801_20170831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170901_20170930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171001_20171031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171101_20171130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171201_20171231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180101_20180131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180201_20180228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180301_20180331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180401_20180430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180501_20180531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180601_20180630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180701_20180731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180801_20180831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180901_20180930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181001_20181031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181101_20181130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181201_20181231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190101_20190131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190201_20190228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190301_20190331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190401_20190430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190501_20190531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190601_20190630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190701_20190731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190801_20190831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190901_20190930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191001_20191031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191101_20191130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191201_20191231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200101_20200131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200201_20200229.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200301_20200331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200401_20200430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200501_20200531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200601_20200630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200701_20200731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200801_20200831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200901_20200930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201001_20201031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201101_20201130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201201_20201231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210101_20210131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210201_20210228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210301_20210331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210401_20210430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210501_20210531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210601_20210630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210701_20210731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210801_20210831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210901_20210930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211001_20211031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211101_20211130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211201_20211231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220101_20220131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220201_20220228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220301_20220331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220401_20220430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220501_20220531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220601_20220630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220701_20220731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220801_20220831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220901_20220930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221001_20221031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221101_20221130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221201_20221231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230101_20230131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230201_20230228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230301_20230331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230401_20230430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230501_20230531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230601_20230630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230701_20230731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230801_20230831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230901_20230930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231001_20231031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231101_20231130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231201_20231231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240101_20240131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240201_20240229.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240301_20240331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240401_20240430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240501_20240531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240601_20240630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240701_20240731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240801_20240831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240901_20240930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241001_20241031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241101_20241130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241201_20241231.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250101_20250131.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250201_20250228.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250301_20250331.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250401_20250430.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250501_20250531.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250601_20250630.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250701_20250731.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250801_20250831.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250901_20250930.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251001_20251031.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251101_20251130.L3m.MO.KD.Kd_490.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251201_20251231.L3m.MO.KD.Kd_490.9km.NRT.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20260101_20260131.L3m.MO.KD.Kd_490.9km.NRT.nc
"""

def download_file_robust(url, filename, session):
    filepath = os.path.join(DOWNLOAD_DIR, filename)
    
    # Resume Logic
    if os.path.exists(filepath):
        # Check if file is valid (>1MB)
        if os.path.getsize(filepath) > 1000000: 
            return "SKIPPED"
        else:
            os.remove(filepath)

    try:
        # 1. Initiate Request (Handle Redirects Manually for Auth)
        response = session.get(url, stream=True)
        
        # If redirected to login page (Earthdata), re-send request to the new URL
        if response.history and "earthdata" in response.url:
             response = session.get(response.url, stream=True)

        response.raise_for_status()
        
        # 2. Download with Progress Bar
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024 # 1MB chunks
        
        with open(filepath, 'wb') as f, tqdm(
            desc=filename,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
            leave=False,
            mininterval=0.5
        ) as bar:
            for chunk in response.iter_content(chunk_size=block_size):
                if chunk:
                    size = f.write(chunk)
                    bar.update(size)
        
        # 3. Validation
        if os.path.getsize(filepath) < 100000: # 100KB check
            return "FAIL_SIZE"
            
        return "SUCCESS"
        
    except Exception as e:
        return f"ERROR: {e}"

def main():
    print("--- NASA Kd_490 Downloader (Local Manual List) ---")
    print(f"Target: {os.path.abspath(DOWNLOAD_DIR)}")
    print(f"Resolution Conversion: {'4km -> 9km' if CONVERT_TO_9KM else 'Keep Original'}")
    
    # Credentials
    print("\nEnter your EarthData Credentials:")
    user = getpass.getpass("Username: ")
    password = getpass.getpass("Password: ")
    
    # SETUP SESSION WITH AUTH
    session = requests.Session()
    # Create Basic Auth Header manually to persist through redirects
    auth_str = f"{user}:{password}"
    b64_auth = base64.b64encode(auth_str.encode()).decode()
    session.headers.update({'Authorization': f"Basic {b64_auth}"})
    
    # Parse Links
    raw_urls = [line.strip() for line in url_list.strip().split('\n') if line.strip()]
    print(f"\nQueueing {len(raw_urls)} files...")
    
    success_count = 0
    
    # Main Loop
    for i, raw_url in enumerate(tqdm(raw_urls, desc="Total Progress")):
        
        url = raw_url
        
        # Feature: Convert 4km links to 9km on the fly
        if CONVERT_TO_9KM and "4km" in url:
            url = url.replace("4km", "9km")
            
        filename = url.split('/')[-1]
        
        # Attempt Download
        status = download_file_robust(url, filename, session)
        
        if status == "SUCCESS":
            success_count += 1
        elif status == "FAIL_SIZE":
            print(f"\n[FAIL] {filename}: File too small (Auth failed)")
        elif "ERROR" in status:
            print(f"\n[FAIL] {filename}: Network Error")
            
        # Polite delay
        time.sleep(0.5)

    print(f"\n--- Done. {success_count}/{len(raw_urls)} files ready. ---")

if __name__ == "__main__":
    main()

--- NASA Kd_490 Downloader (Local Manual List) ---
Target: C:\Users\tejsr\Downloads\BlueEco_Project\data\raw\nasa_kd490_data
Resolution Conversion: 4km -> 9km

Enter your EarthData Credentials:


Username:  ········
Password:  ········



Queueing 282 files...


Total Progress:   0%|          | 0/282 [00:00<?, ?it/s]

AQUA_MODIS.20020801_20020831.L3m.MO.KD.Kd_490.9km.nc:   0%|          | 0.00/4.89M [00:00<?, ?iB/s]

...

AQUA_MODIS.20251001_20251031.L3m.MO.KD.Kd_490.9km.nc:   0%|          | 0.00/4.25M [00:00<?, ?iB/s]

AQUA_MODIS.20251101_20251130.L3m.MO.KD.Kd_490.9km.nc:   0%|          | 0.00/3.97M [00:00<?, ?iB/s]

AQUA_MODIS.20251201_20251231.L3m.MO.KD.Kd_490.9km.NRT.nc:   0%|          | 0.00/3.98M [00:00<?, ?iB/s]

AQUA_MODIS.20260101_20260131.L3m.MO.KD.Kd_490.9km.NRT.nc:   0%|          | 0.00/4.12M [00:00<?, ?iB/s]


--- Done. 282/282 files ready. ---


In [2]:
import os
import requests
import getpass
import time
import base64
from tqdm.notebook import tqdm 

# --- CONFIGURATION ---
# Smart path detection for Local Project Structure
if os.path.exists("../data"):
    DOWNLOAD_DIR = "../data/raw/nasa_poc_data" # <--- CHANGED FOR POC
else:
    DOWNLOAD_DIR = "data/raw/nasa_poc_data"    # <--- CHANGED FOR POC

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# --- SETTINGS ---
# Set this to True to automatically convert any 4km links you paste into 9km links
CONVERT_TO_9KM = True

# --- PASTE YOUR LINKS HERE ---
# Paste your full list of NASA POC URLs between the triple quotes below.
url_list = """
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20020801_20020831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20020901_20020930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021001_20021031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021101_20021130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20021201_20021231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030101_20030131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030201_20030228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030301_20030331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030401_20030430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030501_20030531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030601_20030630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030701_20030731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030801_20030831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20030901_20030930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031001_20031031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031101_20031130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20031201_20031231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040101_20040131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040201_20040229.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040301_20040331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040401_20040430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040501_20040531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040601_20040630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040701_20040731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040801_20040831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20040901_20040930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041001_20041031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041101_20041130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20041201_20041231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050101_20050131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050201_20050228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050301_20050331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050401_20050430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050501_20050531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050601_20050630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050701_20050731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050801_20050831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20050901_20050930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051001_20051031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051101_20051130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20051201_20051231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060101_20060131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060201_20060228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060301_20060331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060401_20060430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060501_20060531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060601_20060630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060701_20060731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060801_20060831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20060901_20060930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061001_20061031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061101_20061130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20061201_20061231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070101_20070131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070201_20070228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070301_20070331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070401_20070430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070501_20070531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070601_20070630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070701_20070731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070801_20070831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20070901_20070930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071001_20071031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071101_20071130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20071201_20071231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080101_20080131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080201_20080229.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080301_20080331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080401_20080430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080501_20080531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080601_20080630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080701_20080731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080801_20080831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20080901_20080930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081001_20081031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081101_20081130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20081201_20081231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090101_20090131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090201_20090228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090301_20090331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090401_20090430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090501_20090531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090601_20090630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090701_20090731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090801_20090831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20090901_20090930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091001_20091031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091101_20091130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20091201_20091231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100101_20100131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100201_20100228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100301_20100331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100401_20100430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100501_20100531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100601_20100630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100701_20100731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100801_20100831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20100901_20100930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101001_20101031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101101_20101130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20101201_20101231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110101_20110131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110201_20110228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110301_20110331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110401_20110430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110501_20110531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110601_20110630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110701_20110731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110801_20110831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20110901_20110930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111001_20111031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111101_20111130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20111201_20111231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120101_20120131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120201_20120229.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120301_20120331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120401_20120430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120501_20120531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120601_20120630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120701_20120731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120801_20120831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20120901_20120930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121001_20121031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121101_20121130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20121201_20121231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130101_20130131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130201_20130228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130301_20130331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130401_20130430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130501_20130531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130601_20130630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130701_20130731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130801_20130831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20130901_20130930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131001_20131031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131101_20131130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20131201_20131231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140101_20140131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140201_20140228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140301_20140331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140401_20140430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140501_20140531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140601_20140630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140701_20140731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140801_20140831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20140901_20140930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141001_20141031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141101_20141130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20141201_20141231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150101_20150131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150201_20150228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150301_20150331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150401_20150430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150501_20150531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150601_20150630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150701_20150731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150801_20150831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20150901_20150930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151001_20151031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151101_20151130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20151201_20151231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160101_20160131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160201_20160229.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160301_20160331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160401_20160430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160501_20160531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160601_20160630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160701_20160731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160801_20160831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20160901_20160930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161001_20161031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161101_20161130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20161201_20161231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170101_20170131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170201_20170228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170301_20170331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170401_20170430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170501_20170531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170601_20170630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170701_20170731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170801_20170831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20170901_20170930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171001_20171031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171101_20171130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20171201_20171231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180101_20180131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180201_20180228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180301_20180331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180401_20180430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180501_20180531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180601_20180630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180701_20180731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180801_20180831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20180901_20180930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181001_20181031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181101_20181130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20181201_20181231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190101_20190131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190201_20190228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190301_20190331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190401_20190430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190501_20190531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190601_20190630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190701_20190731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190801_20190831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20190901_20190930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191001_20191031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191101_20191130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20191201_20191231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200101_20200131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200201_20200229.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200301_20200331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200401_20200430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200501_20200531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200601_20200630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200701_20200731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200801_20200831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20200901_20200930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201001_20201031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201101_20201130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20201201_20201231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210101_20210131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210201_20210228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210301_20210331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210401_20210430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210501_20210531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210601_20210630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210701_20210731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210801_20210831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20210901_20210930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211001_20211031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211101_20211130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20211201_20211231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220101_20220131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220201_20220228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220301_20220331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220401_20220430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220501_20220531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220601_20220630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220701_20220731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220801_20220831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20220901_20220930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221001_20221031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221101_20221130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20221201_20221231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230101_20230131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230201_20230228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230301_20230331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230401_20230430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230501_20230531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230601_20230630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230701_20230731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230801_20230831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20230901_20230930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231001_20231031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231101_20231130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20231201_20231231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240101_20240131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240201_20240229.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240301_20240331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240401_20240430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240501_20240531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240601_20240630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240701_20240731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240801_20240831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20240901_20240930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241001_20241031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241101_20241130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20241201_20241231.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250101_20250131.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250201_20250228.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250301_20250331.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250401_20250430.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250501_20250531.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250601_20250630.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250701_20250731.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250801_20250831.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20250901_20250930.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251001_20251031.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251101_20251130.L3m.MO.POC.poc.9km.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20251201_20251231.L3m.MO.POC.poc.9km.NRT.nc
https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/AQUA_MODIS.20260101_20260131.L3m.MO.POC.poc.9km.NRT.nc
"""

def download_file_robust(url, filename, session):
    filepath = os.path.join(DOWNLOAD_DIR, filename)
    
    # Resume Logic
    if os.path.exists(filepath):
        # Check if file is valid (>1MB)
        if os.path.getsize(filepath) > 1000000: 
            return "SKIPPED"
        else:
            os.remove(filepath)

    try:
        # 1. Initiate Request (Handle Redirects Manually for Auth)
        response = session.get(url, stream=True)
        
        # If redirected to login page (Earthdata), re-send request to the new URL
        if response.history and "earthdata" in response.url:
             response = session.get(response.url, stream=True)

        response.raise_for_status()
        
        # 2. Download with Progress Bar
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024 # 1MB chunks
        
        with open(filepath, 'wb') as f, tqdm(
            desc=filename,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
            leave=False,
            mininterval=0.5
        ) as bar:
            for chunk in response.iter_content(chunk_size=block_size):
                if chunk:
                    size = f.write(chunk)
                    bar.update(size)
        
        # 3. Validation
        if os.path.getsize(filepath) < 100000: # 100KB check
            return "FAIL_SIZE"
            
        return "SUCCESS"
        
    except Exception as e:
        return f"ERROR: {e}"

def main():
    print("--- NASA POC Downloader (Local Manual List) ---")
    print(f"Target: {os.path.abspath(DOWNLOAD_DIR)}")
    print(f"Resolution Conversion: {'4km -> 9km' if CONVERT_TO_9KM else 'Keep Original'}")
    
    # Credentials
    print("\nEnter your EarthData Credentials:")
    user = getpass.getpass("Username: ")
    password = getpass.getpass("Password: ")
    
    # SETUP SESSION WITH AUTH
    session = requests.Session()
    # Create Basic Auth Header manually to persist through redirects
    auth_str = f"{user}:{password}"
    b64_auth = base64.b64encode(auth_str.encode()).decode()
    session.headers.update({'Authorization': f"Basic {b64_auth}"})
    
    # Parse Links
    raw_urls = [line.strip() for line in url_list.strip().split('\n') if line.strip()]
    print(f"\nQueueing {len(raw_urls)} files...")
    
    success_count = 0
    
    # Main Loop
    for i, raw_url in enumerate(tqdm(raw_urls, desc="Total Progress")):
        
        url = raw_url
        
        # Feature: Convert 4km links to 9km on the fly
        if CONVERT_TO_9KM and "4km" in url:
            url = url.replace("4km", "9km")
            
        filename = url.split('/')[-1]
        
        # Attempt Download
        status = download_file_robust(url, filename, session)
        
        if status == "SUCCESS":
            success_count += 1
        elif status == "FAIL_SIZE":
            print(f"\n[FAIL] {filename}: File too small (Auth failed)")
        elif "ERROR" in status:
            print(f"\n[FAIL] {filename}: Network Error")
            
        # Polite delay
        time.sleep(0.5)

    print(f"\n--- Done. {success_count}/{len(raw_urls)} files ready. ---")

if __name__ == "__main__":
    main()

--- NASA POC Downloader (Local Manual List) ---
Target: C:\Users\tejsr\Downloads\BlueEco_Project\data\raw\nasa_poc_data
Resolution Conversion: 4km -> 9km

Enter your EarthData Credentials:


Username:  ········
Password:  ········



Queueing 282 files...


Total Progress:   0%|          | 0/282 [00:00<?, ?it/s]

AQUA_MODIS.20020801_20020831.L3m.MO.POC.poc.9km.nc:   0%|          | 0.00/5.32M [00:00<?, ?iB/s]

...

AQUA_MODIS.20251001_20251031.L3m.MO.POC.poc.9km.nc:   0%|          | 0.00/4.62M [00:00<?, ?iB/s]

AQUA_MODIS.20251101_20251130.L3m.MO.POC.poc.9km.nc:   0%|          | 0.00/4.29M [00:00<?, ?iB/s]

AQUA_MODIS.20251201_20251231.L3m.MO.POC.poc.9km.NRT.nc:   0%|          | 0.00/4.28M [00:00<?, ?iB/s]

AQUA_MODIS.20260101_20260131.L3m.MO.POC.poc.9km.NRT.nc:   0%|          | 0.00/4.46M [00:00<?, ?iB/s]


--- Done. 282/282 files ready. ---


In [1]:
import pandas as pd
import numpy as np
import requests
import io
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import matplotlib.pyplot as plt

def generate_co2_data():
    print("--- 1. DOWNLOADING HISTORY (2002-2025) ---")
    # Official NOAA Mauna Loa Dataset
    url = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.txt"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # Read the file, skipping comment lines starting with '#'
        content = response.text
        # NOAA Columns: year, month, decimal date, average, interpolated, trend, days
        df = pd.read_csv(io.StringIO(content), comment='#', delim_whitespace=True, 
                         names=['year', 'month', 'decimal_date', 'average', 'interpolated', 'trend', 'days'])
        
        # Clean Data
        df = df[df['average'] > 0]  # Remove missing values (-99.99)
        df = df[df['year'] >= 2002] # Start from our project start date
        df = df[df['year'] <= 2025] # Cut off at 'Today' (or end of 2025)
        
        # Create usable Time column
        df['time'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
        df = df[['time', 'year', 'month', 'average']].rename(columns={'average': 'co2_ppm'})
        
        print(f"   [SUCCESS] Downloaded {len(df)} months of real data.")
        print(f"   Start: {df.iloc[0]['time'].date()} = {df.iloc[0]['co2_ppm']} ppm")
        print(f"   End:   {df.iloc[-1]['time'].date()} = {df.iloc[-1]['co2_ppm']} ppm")
        
    except Exception as e:
        print(f"   [FAIL] Could not download from NOAA: {e}")
        return

    print("\n--- 2. TRAINING TREND MODEL (Quadratic) ---")
    # We use a Quadratic Trend (Degree 2) because CO2 is accelerating
    # Formula: CO2 = A * year^2 + B * year + C
    
    # Prepare X (Decimal Year) and y (CO2)
    df['decimal_year'] = df['year'] + (df['month'] - 0.5) / 12
    X = df['decimal_year'].values.reshape(-1, 1)
    y = df['co2_ppm'].values
    
    # Transform X for Polynomial Regression
    poly = PolynomialFeatures(degree=2)
    X_poly = poly.fit_transform(X)
    
    # Train Model
    model = LinearRegression()
    model.fit(X_poly, y)
    print("   [SUCCESS] Model Trained on Historical Data.")

    print("\n--- 3. FORECASTING FUTURE (2026-2035) ---")
    # Create Future Timeline
    future_dates = pd.date_range(start='2026-01-01', end='2035-12-31', freq='ME')
    # Adjust to 1st of month
    future_dates = future_dates + pd.offsets.MonthBegin(0) - pd.offsets.MonthBegin(1) + pd.offsets.Day(1)
    
    future_data = []
    
    # Calculate Average Seasonality from History (to make the forecast look realistic)
    # We calculate the average deviation of each month from the yearly trend
    df['predicted_trend'] = model.predict(X_poly)
    df['seasonality'] = df['co2_ppm'] - df['predicted_trend']
    monthly_seasonality = df.groupby('month')['seasonality'].mean()
    
    for date in future_dates:
        dec_year = date.year + (date.month - 0.5) / 12
        
        # 1. Predict the main Trend line
        trend_val = model.predict(poly.transform([[dec_year]]))[0]
        
        # 2. Add the Seasonality (e.g., CO2 drops in October, peaks in May)
        season_val = monthly_seasonality.loc[date.month]
        
        final_val = trend_val + season_val
        
        future_data.append({
            'time': date,
            'year': date.year,
            'month': date.month,
            'co2_ppm': final_val
        })
        
    df_future = pd.DataFrame(future_data)
    print(f"   [SUCCESS] Generated {len(df_future)} future months.")
    print(f"   2035 Forecast: {df_future.iloc[-1]['co2_ppm']:.2f} ppm")

    print("\n--- 4. MERGING & SAVING ---")
    # Combine History + Future
    cols = ['time', 'year', 'month', 'co2_ppm']
    df_final = pd.concat([df[cols], df_future[cols]]).sort_values('time').reset_index(drop=True)
    
    # Save
    filename = "co2_forcing.csv"
    df_final.to_csv(filename, index=False)
    print(f"   [SAVED] {filename}")
    
    # Validation Plot
    plt.figure(figsize=(10, 6))
    plt.plot(df['time'], df['co2_ppm'], label='NOAA History (Observed)', color='black')
    plt.plot(df_future['time'], df_future['co2_ppm'], label='Forecast (Projected)', color='red', linestyle='--')
    plt.title("CO2 Forcing: History (2002-2025) + Forecast (2026-2035)")
    plt.ylabel("CO2 (ppm)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("co2_projection_plot.png")
    print("   [PLOT] Saved co2_projection_plot.png")

if __name__ == "__main__":
    generate_co2_data()

--- 1. DOWNLOADING HISTORY (2002-2025) ---
   [SUCCESS] Downloaded 0 months of real data.
   [FAIL] Could not download from NOAA: single positional indexer is out-of-bounds


C:\Users\tejsr\AppData\Local\Temp\ipykernel_14548\1846609597.py:21: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(io.StringIO(content), comment='#', delim_whitespace=True,
